### CPE371: Big Data Engineering
### Lab 7.1: Log Analytics with Spark RDDs (Lab 1)

Run the Lab 1 tutorial on `spark.log`, then answer the three Lab 1 questions with RDD code:

| # | question | log source |
| - | -------- | ---------- |
| 1 | How many times does each source appear in the log? | all |
| 2 | Statistics (min, max, avg, outliers, ...) of the python worker | `python.PythonRunner` |
| 3 | Which tasks or stages were lost? | `executor.Executor` |

The same logic is packaged as `src/log_analytics.py` (run `python src/log_analytics.py`) and tested in
`tests/test_log_analytics.py`. The last cell of this notebook checks that both give identical answers.

### Part 0: Spark setup
The lab VM kernel predefines `sc`; locally we create it. The Python workers are pinned to this kernel's interpreter, and `src/` is put on `PYTHONPATH` so they can import the module used in the cross-check.

In [1]:
import os
import re
import sys

# Locate the repo root whether the notebook runs from notebooks/ or the repo root
ROOT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_DIR = os.path.join(ROOT_DIR, "src")
LOG_PATH = os.path.join(ROOT_DIR, "data", "lab7", "spark.log")
sys.path.insert(0, SRC_DIR)
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # workers use this kernel's Python
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [SRC_DIR, os.environ.get("PYTHONPATH")]))

from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("CPE371_Lab7_1_LogAnalytics")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(f"Spark {sc.version} | app {sc.appName} | log {os.path.relpath(LOG_PATH, ROOT_DIR)}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/27 19:47:49 WARN Utils: Your hostname, MacBookAirPiti-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
26/09/27 19:47:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/27 19:47:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | app CPE371_Lab7_1_LogAnalytics | log data/lab7/spark.log


### Part 1: Lab 1 tutorial (Map and Reduce, pair RDDs)
The original tutorial cells, run against the local copy of `spark.log`.

In [2]:
sc.parallelize("a b c d").flatMap(lambda x: x.split(" ")[0]).take(4)

['a', 'b', 'c', 'd']

In [3]:
lines = sc.textFile(LOG_PATH)
lines.take(3)

['17/06/09 20:10:40 INFO executor.CoarseGrainedExecutorBackend: Registered signal handlers for [TERM, HUP, INT]',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing modify acls to: yarn,curi']

In [4]:
lineLengths = lines.map(lambda s: len(s))
totalLength = lineLengths.reduce(lambda a, b: a + b)
print(lineLengths.take(5), totalLength)

[109, 78, 80, 198, 78] 192268


In [5]:
log = sc.textFile(LOG_PATH)
print("lines:", log.count())
print(log.flatMap(lambda x: x.split()).take(15))

lines: 2000
['17/06/09', '20:10:40', 'INFO', 'executor.CoarseGrainedExecutorBackend:', 'Registered', 'signal', 'handlers', 'for', '[TERM,', 'HUP,', 'INT]', '17/06/09', '20:10:40', 'INFO', 'spark.SecurityManager:']


In [6]:
date = log.map(lambda x: x.split(' ')[0])
date.take(3)

['17/06/09', '17/06/09', '17/06/09']

In [7]:
source_info_rdd = log.map(lambda x: (x.split()[3], " ".join(x.split()[4:])))
source_info_rdd.take(3)

[('executor.CoarseGrainedExecutorBackend:',
  'Registered signal handlers for [TERM, HUP, INT]'),
 ('spark.SecurityManager:', 'Changing view acls to: yarn,curi'),
 ('spark.SecurityManager:', 'Changing modify acls to: yarn,curi')]

In [8]:
source_len_rdd = source_info_rdd.map(lambda x: (x[0], len(x[1].split())))
source_nword_rdd = source_len_rdd.reduceByKey(lambda x, y: x + y)
source_nline_rdd = source_len_rdd.groupByKey().mapValues(lambda x: len(x))
source_nline_rdd.join(source_nword_rdd).take(5)

[('spark.SecurityManager:', (6, 56)),
 ('Remoting:', (2, 8)),
 ('storage.DiskBlockManager:', (1, 5)),
 ('storage.MemoryStore:', (150, 2092)),
 ('executor.Executor:', (606, 6647))]

In [9]:
source_avglen_rdd = source_len_rdd.groupByKey().mapValues(lambda x: sum(x) / len(x))
source_avglen_rdd.sortBy(lambda x: x[1]).take(5)

[('mapred.SparkHadoopMapRedUtil:', 2.0),
 ('slf4j.Slf4jLogger:', 2.0),
 ('storage.BlockManagerMaster:', 3.0),
 ('rdd.HadoopRDD:', 3.0),
 ('Remoting:', 4.0)]

In [10]:
print("partitions:", source_avglen_rdd.getNumPartitions(), "->", source_avglen_rdd.repartition(2).getNumPartitions())
print("mean words per line:", round(source_len_rdd.values().mean(), 2))

partitions: 2 -> 2
mean words per line: 8.76


---
### Question 1: How many times does each source appear in the log?

**Approach.** Every line has the layout `date time LEVEL source: message`, so the source is field 3.

1. Drop lines with fewer than 4 fields. `wc -l` reports 1999 lines because the last line has no trailing newline, and Spark reads 2000. None of them is blank here, but the guard keeps a truncated line from raising `IndexError`.
2. `map` each line to `(source, 1)`, stripping the trailing `:`.
3. `reduceByKey(+)` sums per source. This combines on each partition before the shuffle, which makes it cheaper than `groupByKey`.
4. `sortBy(-count, source)` puts the most frequent first, with a deterministic order for ties.

In [11]:
log = sc.textFile(LOG_PATH).filter(lambda line: len(line.split()) >= 4).cache()

source_counts = (log.map(lambda line: (line.split()[3].rstrip(":"), 1))
                    .reduceByKey(lambda a, b: a + b)
                    .sortBy(lambda kv: (-kv[1], kv[0])))

q1 = source_counts.collect()
spark.createDataFrame(q1, ["source", "count"]).show(truncate=False)
print(f"{len(q1)} distinct sources, {sum(c for _, c in q1)} lines in total")

+-------------------------------------+-----+
|source                               |count|
+-------------------------------------+-----+
|executor.Executor                    |606  |
|python.PythonRunner                  |375  |
|executor.CoarseGrainedExecutorBackend|308  |
|storage.BlockManager                 |257  |
|storage.MemoryStore                  |150  |
|spark.CacheManager                   |75   |
|broadcast.TorrentBroadcast           |74   |
|output.FileOutputCommitter           |60   |
|rdd.HadoopRDD                        |45   |
|mapred.SparkHadoopMapRedUtil         |30   |
|spark.SecurityManager                |6    |
|Configuration.deprecation            |5    |
|Remoting                             |2    |
|storage.BlockManagerMaster           |2    |
|util.Utils                           |2    |
|netty.NettyBlockTransferService      |1    |
|slf4j.Slf4jLogger                    |1    |
|storage.DiskBlockManager             |1    |
+---------------------------------

**Answer.** There are 18 distinct sources across 2000 lines. `executor.Executor` is the most frequent (606 lines), followed by `python.PythonRunner` (375) and `executor.CoarseGrainedExecutorBackend` (308). The example source from the brief, `spark.SecurityManager`, appears **6** times, all during executor start-up.

---
### Question 2: Statistics of the python worker (`python.PythonRunner`)

**Approach.** Every PythonRunner line reports one Python worker's timings in milliseconds:
`Times: total = 1072, boot = 856, init = 210, finish = 6`.

1. `filter` on the source, then parse the four numbers with a regex. The regex must accept a **negative** `boot`, as explained below.
2. `RDD.stats()` returns count, mean and stdev in a single pass (a `StatCounter`).
3. Spark RDDs have no percentile function. We `sortBy` the values and `collect` them, which is only 375 numbers, to get min, max and the quartiles.
4. **Outliers** use the Tukey rule: a value is an outlier when it lies outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`. The outliers themselves are then selected with an RDD `filter`.
5. **Severity.** The IQR of `total` is only 3 ms, so the fences are very tight. Each outlier is labelled **severe** when it is more than 10× the median, otherwise **mild**. This way the count of 46 does not hide which runs are actually slow.

In [12]:
TIMES = re.compile(r"Times: total = (-?\d+), boot = (-?\d+), init = (-?\d+), finish = (-?\d+)")
FIELDS = ["total", "boot", "init", "finish"]

def parse_times(line):
    m = TIMES.search(line)
    return dict(zip(FIELDS, map(int, m.groups()))) if m else None

times = (log.filter(lambda line: line.split()[3] == "python.PythonRunner:")
            .map(parse_times)
            .filter(lambda t: t is not None)
            .cache())
print(times.count(), "PythonRunner records, e.g.", times.first())

375 PythonRunner records, e.g. {'total': 1072, 'boot': 856, 'init': 210, 'finish': 6}


In [13]:
def quantile(sorted_values, q):
    pos = (len(sorted_values) - 1) * q
    lo = int(pos); hi = min(lo + 1, len(sorted_values) - 1)
    return sorted_values[lo] + (sorted_values[hi] - sorted_values[lo]) * (pos - lo)

rows = []
for f in FIELDS:
    values = times.map(lambda t, f=f: t[f])
    s = values.stats()                                   # count, min, max, mean, stdev in one pass
    ordered = values.sortBy(lambda v: v).collect()
    p25, p50, p75 = (quantile(ordered, q) for q in (0.25, 0.5, 0.75))
    lo, hi = p25 - 1.5 * (p75 - p25), p75 + 1.5 * (p75 - p25)
    n_out = values.filter(lambda v, lo=lo, hi=hi: v < lo or v > hi).count()
    # min/max from the sorted list: StatCounter gives numpy floats, which createDataFrame rejects
    rows.append((f, int(s.count()), ordered[0], ordered[-1], round(float(s.mean()), 2), round(float(s.stdev()), 2),
                 p25, p50, p75, lo, hi, n_out))

q2 = spark.createDataFrame(rows, ["field", "count", "min", "max", "avg", "stdev",
                                  "q1", "median", "q3", "lower_fence", "upper_fence", "outliers"])
q2.show()

+------+-----+-----+----+-------+-------+-----+------+-----+-----------+-----------+--------+
| field|count|  min| max|    avg|  stdev|   q1|median|   q3|lower_fence|upper_fence|outliers|
+------+-----+-----+----+-------+-------+-----+------+-----+-----------+-----------+--------+
| total|  375|   37|1114|  55.66| 119.56| 39.0|  41.0| 42.0|       34.5|       46.5|      46|
|  boot|  375|-8772| 885|-216.69|1112.47|-96.0|   8.0| 18.0|     -267.0|      189.0|      35|
|  init|  375|    4|8827| 271.31| 1106.3| 22.0|  40.0|141.0|     -156.5|      319.5|      30|
|finish|  375|    0|  10|   1.03|   1.72|  0.0|   0.0|  1.0|       -1.5|        2.5|      45|
+------+-----+-----+----+-------+-------+-----+------+-----+-----------+-----------+--------+



In [14]:
total_stats = [r for r in rows if r[0] == "total"][0]
lo, hi = total_stats[9], total_stats[10]
# The IQR is only 3 ms, so the fences also catch runs that are just a little slow.
# Severity separates them: "severe" above 10x the median, otherwise "mild".
severe_above = 10 * total_stats[7]
outliers = (times.filter(lambda t: t["total"] < lo or t["total"] > hi)
                 .map(lambda t: dict(t, severity="severe" if t["total"] > severe_above else "mild"))
                 .sortBy(lambda t: -t["total"]))
print(f"{outliers.count()} outliers on total (outside {lo} .. {hi} ms):",
      dict(outliers.map(lambda t: (t["severity"], 1)).reduceByKey(lambda a, b: a + b).collect()))
spark.createDataFrame(outliers.take(10)).select(*FIELDS, "severity").show()

negative_boot = times.filter(lambda t: t["boot"] < 0)
print(negative_boot.count(), "records have a negative boot time, e.g.", negative_boot.take(2))
mismatch = times.filter(lambda t: t["total"] != t["boot"] + t["init"] + t["finish"]).count()
print(mismatch, "records where total != boot + init + finish")

46 outliers on total (outside 34.5 .. 46.5 ms): {'severe': 5, 'mild': 41}
+-----+-----+----+------+--------+
|total| boot|init|finish|severity|
+-----+-----+----+------+--------+
| 1114|  885| 223|     6|  severe|
| 1078|  851| 219|     8|  severe|
| 1077|  865| 206|     6|  severe|
| 1074|  869| 199|     6|  severe|
| 1072|  856| 210|     6|  severe|
|  108|   28|  79|     1|    mild|
|   65|   10|  45|    10|    mild|
|   60|-8772|8827|     5|    mild|
|   59|-8772|8826|     5|    mild|
|   58|    7|  46|     5|    mild|
+-----+-----+----+------+--------+

169 records have a negative boot time, e.g. [{'total': 42, 'boot': -4131, 'init': 4172, 'finish': 1}, {'total': 41, 'boot': -4122, 'init': 4162, 'finish': 1}]


0 records where total != boot + init + finish


**Answer.** 375 python worker runs were logged.

| field | min | max | avg | median | outliers (1.5 IQR) |
| ----- | --- | --- | --- | ------ | ------------------ |
| total | 37 ms | 1114 ms | 55.66 ms | 41 ms | 46 (5 severe, 41 mild) |
| boot | -8772 ms | 885 ms | -216.69 ms | 8 ms | 35 |
| init | 4 ms | 8827 ms | 271.31 ms | 40 ms | 30 |
| finish | 0 ms | 10 ms | 1.03 ms | 0 ms | 45 |

What the numbers show:

1. **A typical worker takes about 41 ms.** The interquartile range of `total` is only 39–42 ms, so the fences are tight (34.5–46.5 ms) and 46 runs fall outside them. **41 of the 46 are mild,** at 47–108 ms. They count as outliers only because the normal range is so narrow, and they are not a performance problem.
2. **The 5 severe outliers are cold starts,** at 1072–1114 ms. They are the first 5 tasks of stage 0 at 20:10:48, where `boot` alone takes about 850–885 ms because a fresh Python process has to start. That is 20× the median. After that, the daemon reuses workers.
3. **The mean is misleading.** 55.66 ms is well above the 41 ms median because those 5 cold starts pull it up. The median is the better summary here.
4. **169 records (45%) have a negative `boot`,** down to -8772 ms. A duration cannot be negative. In every record, `total = boot + init + finish` holds exactly (checked below). So the time is not missing, only misattributed: `boot = -8772` pairs with `init = 8827`, and `total` is still a normal 60 ms. The likely cause is a reused, daemon-forked worker whose boot start time predates the task. `total` is therefore the reliable field, and the `boot` and `init` statistics are a data-quality finding rather than performance figures.

---
### Question 3: Which tasks or stages were lost? (`executor.Executor`)

**Approach.** The log has no explicit `Lost task` or `ERROR` lines. `executor.Executor` writes only three kinds of message:
`Running task T in stage S (TID n)`, `Finished task T in stage S (TID n)` and a single `Starting executor`.
So a **lost task** is one that has a `Running` line but no matching `Finished` line. The TID is unique per task attempt, so it is the join key.

1. Parse each executor line into a pair RDD `(tid, (event, task, stage, time))`.
2. `subtractByKey(finished)` on the started tasks leaves the ones that never finished.
3. For stages, `reduceByKey` sums `(started, finished)` per stage. A stage with `started > finished` never completed.

In [15]:
TASK = re.compile(r"(Running|Finished) task (\d+\.\d+) in stage (\d+\.\d+) \(TID (\d+)\)")

def parse_task(line):
    m = TASK.search(line)
    if not m:
        return None
    event, task, stage, tid = m.groups()
    return int(tid), (event, task, stage, " ".join(line.split()[:2]))

executor = log.filter(lambda line: line.split()[3] == "executor.Executor:")
print(executor.map(lambda l: re.sub(r"\d+", "N", " ".join(l.split()[4:7]))).countByValue())

events = executor.map(parse_task).filter(lambda e: e is not None).cache()
started = events.filter(lambda e: e[1][0] == "Running")
finished = events.filter(lambda e: e[1][0] == "Finished")
print("started:", started.count(), "| finished:", finished.count())

defaultdict(<class 'int'>, {'Starting executor ID': 1, 'Running task N.N': 305, 'Finished task N.N': 300})
started: 305 | finished: 300


In [16]:
lost = started.subtractByKey(finished).sortByKey()
spark.createDataFrame(lost.map(lambda e: (e[0], e[1][1], e[1][2], e[1][3])).collect(),
                      ["tid", "task", "stage", "started_at"]).show()

+----+----+-----+-----------------+
| tid|task|stage|       started_at|
+----+----+-----+-----------------+
|1350|30.0| 29.0|17/06/09 20:11:11|
|1351|31.0| 29.0|17/06/09 20:11:11|
|1352|32.0| 29.0|17/06/09 20:11:11|
|1353|33.0| 29.0|17/06/09 20:11:11|
|1354|34.0| 29.0|17/06/09 20:11:11|
+----+----+-----+-----------------+



In [17]:
per_stage = (events.map(lambda e: (e[1][2], (1, 0) if e[1][0] == "Running" else (0, 1)))
                   .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                   .map(lambda kv: (kv[0], kv[1][0], kv[1][1], kv[1][0] - kv[1][1]))
                   .sortBy(lambda r: float(r[0])))
stage_df = spark.createDataFrame(per_stage.collect(), ["stage", "started", "finished", "lost"])
print(stage_df.count(), "stages seen; stages with lost tasks:")
stage_df.filter("lost > 0").show()
print("last lines of the log:")
for line in sc.textFile(LOG_PATH).collect()[-3:]:
    print(" ", line)

30 stages seen; stages with lost tasks:
+-----+-------+--------+----+
|stage|started|finished|lost|
+-----+-------+--------+----+
| 29.0|     35|      30|   5|
+-----+-------+--------+----+

last lines of the log:
  17/06/09 20:11:11 INFO executor.CoarseGrainedExecutorBackend: Got assigned task 1354
  17/06/09 20:11:11 INFO executor.Executor: Running task 34.0 in stage 29.0 (TID 1354)
  17/06/09 20:11:11 INFO storage.BlockManager: Found block rdd_42_32 locally


**Answer.** 305 tasks started and 300 finished, so **5 tasks were lost**: TIDs **1350–1354**, which are tasks 30.0–34.0 of **stage 29.0**. All five started at 17/06/09 20:11:11.

Stages 0.0–28.0 all finished every task they started, so **stage 29.0 is the only lost stage.** Its tasks 0–29 finished, but 30–34 never did.

The last log line is the start of task 34.0 at the same second. The executor log simply stops there, with no error or exception. That points to the executor being killed or the log being cut off (for example, the YARN container ending), not to a task failing inside Spark. On a real cluster the driver would mark these tasks as lost and reschedule them on another executor.

---
### Cross-check with `src/log_analytics.py`
The packaged module must give the same three answers as the step-by-step code above.

In [18]:
import log_analytics as la

assert la.source_counts(log) == q1
assert la.python_worker_statistics(log)["total"]["outliers"] == total_stats[-1]
assert la.outlier_severity_counts(la.python_worker_outliers(log)) ==     dict(outliers.map(lambda t: (t["severity"], 1)).reduceByKey(lambda a, b: a + b).collect())
assert [t["tid"] for t in la.lost_tasks(log)] == lost.keys().collect()
assert la.lost_stages(log) == [r.stage for r in stage_df.filter("lost > 0").collect()]
print("module and notebook agree on Q1, Q2 and Q3")

module and notebook agree on Q1, Q2 and Q3


In [19]:
spark.stop()